In [0]:
import mlflow
from mlflow.tracking import MlflowClient

client = MlflowClient()

# 1. Obtener el experiment_id del notebook actual en Databricks
try:
    # Intenta obtener el ID del experimento ligado al notebook
    experiment_id = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookId().get()
except Exception:
    # Si estás ejecutando fuera del notebook estándar, usa el experimento activo o default
    current_experiment = mlflow.get_experiment_by_name(
        dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
    )
    experiment_id = current_experiment.experiment_id

print(f"Experimento detectado ID: {experiment_id}")

# 2. Buscar la mejor corrida según tu métrica
# Cambia 'metrics.accuracy DESC' o 'metrics.rmse ASC' según tu proyecto
runs = mlflow.search_runs(
    experiment_ids=[str(experiment_id)],
    order_by=["metrics.accuracy DESC"], 
    max_results=1
)

if not runs.empty:
    best_run = runs.iloc[0]
    best_run_id = best_run["run_id"]
    print(f"Mejor Run ID: {best_run_id}")

    # 3. Registrar el modelo en MLflow Model Registry
    model_name = "Modelo_MLOps"
    model_uri = f"runs:/{best_run_id}/model"
    registered_model = mlflow.register_model(model_uri=model_uri, name=model_name)

    # 4. Asignar el alias PRINCIPAL
    client.set_registered_model_alias(
        name=model_name,
        alias="PRINCIPAL",
        version=registered_model.version
    )
    print(f"✅ Alias 'PRINCIPAL' asignado exitosamente a la Versión {registered_model.version} del modelo {model_name}.")
else:
    print("⚠️ No se encontraron ejecuciones (runs) en este experimento. Asegúrate de ejecutar primero el entrenamiento con mlflow.start_run().")